# DP-HPO Experiment — UCI Breast Cancer

**Dataset:** UCI Breast Cancer  
**Estimated time:** ~20–30 minutes  
**Output:** `results_v2_breast_cancer.json`

> Before running: Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install -q optuna scikit-optimize scipy scikit-learn numpy
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!wget -q https://raw.githubusercontent.com/kartikjsonawane/dp-hpo/master/dp_hpo.py
import dp_hpo
print('dp_hpo.py loaded OK')

In [ ]:
import numpy as np, json, time, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from scipy.stats import wilcoxon as scipy_wilcoxon
from dp_hpo import (run_all_methods, DIMS, VALS, DEFAULTS)

N_SEEDS      = 25
SUBSAMPLE    = None
N_BASELINES  = 8
SEEDS        = list(range(N_SEEDS))
BONFERRONI_M = 32   # 8 baselines × 4 datasets
ALPHA_ADJ    = 0.05 / BONFERRONI_M
METHODS_LIST = [
    'Grid Search','Random Search (k=20)','Random Search (k=10)',
    'Bayesian Opt','Optuna (TPE)','Hyperband','BOHB','SMAC','DP-HPO (Proposed)'
]
BASELINES = [m for m in METHODS_LIST if m != 'DP-HPO (Proposed)']

def prep(X, y, seed, subsample=None):
    if subsample and len(X) > subsample:
        idx = np.random.RandomState(seed).choice(len(X), subsample, replace=False)
        X, y = X[idx], y[idx]
    Xr, Xv, yr, yv = train_test_split(X, y, test_size=0.2,
                                       random_state=seed, stratify=y)
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    Xr = sc.fit_transform(Xr).astype(np.float32)
    Xv = sc.transform(Xv).astype(np.float32)
    return Xr, yr, Xv, yv

def wilcoxon_test(accs_a, accs_b, name_b='baseline'):
    diffs = np.array(accs_a) - np.array(accs_b)
    if np.all(diffs == 0):
        return {'W':None,'p':None,'significant':False,'effect_size_d':0.0}
    stat, p = scipy_wilcoxon(accs_a, accs_b, alternative='two-sided')
    d = float(np.mean(diffs)/(np.std(diffs)+1e-12))
    sig = p < ALPHA_ADJ
    print(f'  vs {name_b:<28} p={p:.4f}  d={d:+.3f}  {"SIG" if sig else "ns"}')
    return {'W':float(stat),'p':float(p),'significant':bool(sig),'effect_size_d':d}

print('Config ready.')

In [ ]:
# Load dataset
X, y = load_breast_cancer(return_X_y=True)
print(f'Breast Cancer: {X.shape}')


In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────
ds_results = {m: {'accs':[],'evals':[],'times':[]} for m in METHODS_LIST}
wall_start = time.time()

for seed in SEEDS:
    t0 = time.time()
    print(f'Seed {seed+1}/{N_SEEDS} ...', end=' ', flush=True)
    Xr, yr, Xv, yv = prep(X, y, seed, SUBSAMPLE)
    run = run_all_methods(Xr, yr, Xv, yv, random_state=seed)
    for method, vals in run.items():
        if method in ds_results:
            ds_results[method]['accs'].append(vals['accuracy'])
            ds_results[method]['evals'].append(vals['n_evaluations'])
            ds_results[method]['times'].append(vals['time_seconds'])
    elapsed = time.time() - t0
    dp_acc = ds_results['DP-HPO (Proposed)']['accs'][-1]
    print(f'done in {elapsed:.0f}s | DP-HPO acc={dp_acc:.4f}')

total_hrs = (time.time() - wall_start) / 3600
print(f'\nAll {N_SEEDS} seeds done in {total_hrs:.2f} hours')

In [ ]:
# ── Summary + Wilcoxon ───────────────────────────────────────────────
print('\n── UCI Breast Cancer Results ─────────────────────────────────────────')
print(f"  {'Method':<32} {'Mean%':>8} {'Std%':>6} {'Evals':>6}")
print('  ' + '-'*58)
for m in METHODS_LIST:
    v = ds_results[m]
    if not v['accs']: continue
    print(f"  {m:<32} {np.mean(v['accs'])*100:>7.2f}%  "
          f"{np.std(v['accs'])*100:>5.2f}  {np.mean(v['evals']):>6.0f}")

print(f'\nWilcoxon vs DP-HPO (α_adj={ALPHA_ADJ:.5f}):')
dp_accs = ds_results['DP-HPO (Proposed)']['accs']
sig_results = {}
for b in BASELINES:
    if ds_results[b]['accs']:
        sig_results[b] = wilcoxon_test(dp_accs, ds_results[b]['accs'], name_b=b)

In [ ]:
# ── Save partial JSON ────────────────────────────────────────────────
partial = {
    'dataset': 'UCI Breast Cancer',
    'n_seeds': N_SEEDS,
    'seeds':   SEEDS,
    'bonferroni_m': BONFERRONI_M,
    'results': {
        m: {
            'accs':       [float(x) for x in v['accs']],
            'evals':      [int(x)   for x in v['evals']],
            'times':      [float(x) for x in v['times']],
            'mean_acc':   float(np.mean(v['accs']))  if v['accs'] else None,
            'std_acc':    float(np.std(v['accs']))   if v['accs'] else None,
            'mean_evals': float(np.mean(v['evals'])) if v['evals'] else None,
            'mean_time':  float(np.mean(v['times'])) if v['times'] else None,
        }
        for m, v in ds_results.items()
    },
    '_wilcoxon': sig_results,
}

with open('results_v2_breast_cancer.json', 'w') as f:
    json.dump(partial, f, indent=2)
print('Saved: results_v2_breast_cancer.json')

from google.colab import files
files.download('results_v2_breast_cancer.json')
print('Download triggered!')